In [1]:
import PyPDF2

print("PyPDF2 version:", PyPDF2.__version__)

PyPDF2 version: 3.0.1


In [2]:
pdf_file_path = "handbook-of-academic-regulations-2025-2026.pdf"

reader = PyPDF2.PdfReader(pdf_file_path)
print(f"The PDF has {len(reader.pages)} pages.")

# read first page
first_page = reader.pages[0]
text = first_page.extract_text()
print(text[:1200])

The PDF has 166 pages.
 Handbook of Academic Regulations 2025  
 
  
 
  
 
  
 
  
 
  
 
  
 
 
 
 
 
  
 
 
 
 
  
 
 
  
 
 
 
 
 
 
 
 
 
 
   
 
Owned By:  Academic Council  
Maintained by:  Academic Registrar’s Department  
Last updated:  April 2025  
Approved on:  June 2025  
Effective from:  1 August 2025 
Review date:  April 2026
 


In [3]:
# Extract all pages
all_text = ""

for page in reader.pages:
    page_text = page.extract_text()
    all_text += page_text + "\n"

print("✅ Total characters extracted:", len(all_text))



✅ Total characters extracted: 483614


In [4]:
# Inspect a slice of the raw extract (avoid huge prints)
print(all_text[:4000])
print("\n--- ... ---\n")
print(all_text[len(all_text) // 2 : len(all_text) // 2 + 2000])


 Handbook of Academic Regulations 2025  
 
  
 
  
 
  
 
  
 
  
 
  
 
 
 
 
 
  
 
 
 
 
  
 
 
  
 
 
 
 
 
 
 
 
 
 
   
 
Owned By:  Academic Council  
Maintained by:  Academic Registrar’s Department  
Last updated:  April 2025  
Approved on:  June 2025  
Effective from:  1 August 2025 
Review date:  April 2026
 
Part 1: Context  
Section 1: Introduction 
1.1. Thi
s handbook contains all the academic regulations for taught courses leading to awards of 
the University of Westminster delivered on the University’s campuses and through distance 
learning and collaborative provision. These are set out in Parts 1 to 6 of this handbook.  
1.2. The 
regulations and processes, which govern research degrees, are set out in the Research 
Degree Regulations and Handbook  and the Research Framework . 
1.3. The 
following documents also contain valuable information and can be accessed online:  
Quali
ty Assurance and Enhancement Handbook 
Student Charter   
Student Code of Conduct   
Universit

In [5]:
import re
import unicodedata

DOCUMENT_TITLE = "Handbook of Academic Regulations 2025–2026"


def fix_broken_clause_numbers(t: str) -> str:
    """Repair clause numbers split across lines (e.g. '1.\\n11.' → '1.11.', orphan '1\\n1.10.' → '1.10.')."""
    t = re.sub(r"(?m)^(\d{1,2})\.\s*\n\s*(\d{1,3}\.\s+)", r"\1.\2", t)
    lines = t.splitlines()
    out = []
    i = 0
    while i < len(lines):
        cur = lines[i].strip()
        if i + 1 < len(lines):
            nxt = lines[i + 1].lstrip()
            m = re.match(r"^(\d{1,2})\.(\d{1,3})\.\s", nxt)
            if m and cur.isdigit() and cur == m.group(1):
                i += 1
                continue
        out.append(lines[i])
        i += 1
    return "\n".join(out)


def clean_english_handbook_text(text: str) -> str:
    """Page noise removal, clause fixes, hyphen/spacing normalization, PDF line glue."""
    t = unicodedata.normalize("NFKC", text)
    t = t.replace("\uFFFD", "'")
    for ch in ("\u00a0", "\u202f", "\u2009", "\u2007"):
        t = t.replace(ch, " ")

    t = fix_broken_clause_numbers(t)

    # Spaced hyphenated compounds: full - time → full-time
    t = re.sub(r"\b([A-Za-z]{2,})\s*-\s*([A-Za-z]{2,})\b", r"\1-\2", t)

    t = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", t)
    t = re.sub(r"([a-z])\n([a-z])", r"\1\2", t)

    out_lines = []
    for ln in t.splitlines():
        ln = ln.strip()
        if not ln:
            continue
        # Drop standalone page numbers / page labels (footer noise)
        if re.fullmatch(r"\d{1,3}", ln):
            continue
        if re.fullmatch(r"Page\s+\d+(?:\s+of\s+\d+)?", ln, flags=re.IGNORECASE):
            continue
        ln = re.sub(r"[ \t]+", " ", ln)
        out_lines.append(ln)
    t = "\n".join(out_lines)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


cleaned_text = clean_english_handbook_text(all_text)
print("✅ Total characters after cleaning:", len(cleaned_text))
print(cleaned_text[:900])


✅ Total characters after cleaning: 468811
Handbook of Academic Regulations 2025
Owned By: Academic Council
Maintained by: Academic Registrar’s Department
Last updated: April 2025
Approved on: June 2025
Effective from: 1 August 2025
Review date: April 2026
Part 1: Context
Section 1: Introduction
1.1. This handbook contains all the academic regulations for taught courses leading to awards of
the University of Westminster delivered on the University’s campuses and through distance
learning and collaborative provision. These are set out in Parts 1 to 6 of this handbook.
1.2. The
regulations and processes, which govern research degrees, are set out in the Research
Degree Regulations and Handbook and the Research Framework .
1.3. The
following documents also contain valuable information and can be accessed online:
Quality Assurance and Enhancement Handbook
Student Charter
Student Code of Conduct
University Calendar
Collaborations Hand


In [6]:
# Cleaning is done in the previous cell. Quick sanity check:
print("✅ Ready for save / chunking.")
print(cleaned_text[:600])


✅ Ready for save / chunking.
Handbook of Academic Regulations 2025
Owned By: Academic Council
Maintained by: Academic Registrar’s Department
Last updated: April 2025
Approved on: June 2025
Effective from: 1 August 2025
Review date: April 2026
Part 1: Context
Section 1: Introduction
1.1. This handbook contains all the academic regulations for taught courses leading to awards of
the University of Westminster delivered on the University’s campuses and through distance
learning and collaborative provision. These are set out in Parts 1 to 6 of this handbook.
1.2. The
regulations and processes, which govern research degrees, ar


In [7]:
from pathlib import Path

DATA_PROCESSED = Path("data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

save_path = DATA_PROCESSED / "academic_handbook_2025_26_cleaned.txt"

with open(save_path, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print("✅ Saved cleaned handbook to:", save_path.resolve())


✅ Saved cleaned handbook to: C:\Users\Hasan Jaafar\Documents\finalyearproject\final submission\chatbot\chatbot\data\processed\academic_handbook_2025_26_cleaned.txt


In [8]:
# Chunk by scanning lines: track Part / Section state; clause lines start a new chunk.
# Part/Section headings are NOT appended to chunk text (avoids next heading inside prior clause).
# Standalone PDF subheadings (title-style lines) flush the current chunk before the line is appended,
# only if the previous line ends a sentence (so lists after “:” are not split).
PART_LINE = re.compile(r"^Part\s+(\d+)\s*:\s*(.*)$", re.IGNORECASE)
SECTION_LINE = re.compile(r"^Section\s+(\d+)\s*:\s*(.*)$", re.IGNORECASE)
CLAUSE_LINE = re.compile(r"^(\d{1,2}\.\d{1,3})\.\s+(.*)$")


def looks_like_standalone_subheading(line: str) -> bool:
    """Standalone title lines (not Part/Section/clause) — flush chunk so they are not tacked onto the prior clause."""
    s = line.strip()
    if len(s) < 6 or len(s) > 140:
        return False
    if not s[0].isupper():
        return False
    if CLAUSE_LINE.match(s) or PART_LINE.match(s) or SECTION_LINE.match(s):
        return False
    if s.startswith("Note:"):
        return False
    if re.match(r"^Table\s+\d+", s, re.I):
        return False
    if re.match(r"^[a-z]\)\s", s, re.I):
        return False
    if re.match(r"^\d+[-–.)]\s", s):
        return False
    if s.endswith(".") and len(s) > 85:
        return False
    words = s.split()
    if len(words) < 2 or len(words) > 18:
        return False
    if s.count(",") > 2:
        return False
    if len(words) > 8 and re.search(
        r"\b(will|shall|must|should|may|might|can|could|would|is|are|was|were|has|have|had)\b",
        s,
        re.I,
    ):
        return False
    small = {"a",
        "an",
        "the",
        "of",
        "in",
        "for",
        "to",
        "and",
        "or",
        "on",
        "at",
        "by",
        "with",
        "via",
        "as",
    }

    def titleish_word(w: str) -> bool:
        w = w.strip("’'\"“”")
        if not w:
            return False
        if w.lower() in small:
            return True
        return w[0].isupper()

    if sum(1 for w in words if titleish_word(w)) / len(words) >= 0.6:
        return True
    if re.match(
        r"^(Claims|Principles|Outcomes|Procedures|Guidance|Fulfilment|Collaborations|"
        r"Assessment|Sitting|Introduction|Definitions|Scope)\b",
        s,
        re.I,
    ):
        return True
    return False


def _last_line_closes_sentence(last: str) -> bool:
    """Avoid splitting mid-clause lists (e.g. document names after a colon) — only flush after a real sentence end."""
    s = last.rstrip()
    if not s:
        return False
    while s and s[-1] in "\"'":
        s = s[:-1].rstrip()
    if not s:
        return False
    if s[-1] in ".!?":
        return True
    if s[-1] == ")" and len(s) >= 2 and s[-2] in ".!?":
        return True
    return False


def _meta_dict(part: str, section: str, clause: str) -> dict:
    """Strings only (Chroma-friendly). Align with Meta_data / ingest: clause + is_continuation."""
    return {
        "document": DOCUMENT_TITLE,
        "part": part or "",
        "section": section or "",
        "clause": clause or "",
        "is_continuation": False,
    }


def chunk_by_clauses_stateful(cleaned: str) -> list:
    current_part = ""
    current_section = ""
    chunks_out = []
    buf: list[str] = []
    current_clause = None  # last clause id for chunk body; None = preamble / non-clause block

    def flush():
        nonlocal buf, current_clause
        if not buf:
            return
        text = "\n".join(buf).strip()
        if not text:
            buf = []
            return
        chunks_out.append(
            {
                "text": text,
                "metadata": _meta_dict(current_part, current_section, current_clause or ""),
            }
        )
        buf = []

    for line in cleaned.splitlines():
        line = line.rstrip()
        if not line:
            continue

        pm = PART_LINE.match(line)
        if pm:
            flush()
            current_part = pm.group(1)
            current_section = ""
            continue

        sm = SECTION_LINE.match(line)
        if sm:
            flush()
            current_section = sm.group(1)
            continue

        cm = CLAUSE_LINE.match(line)
        if cm:
            flush()
            current_clause = cm.group(1)
            rest = cm.group(2).strip()
            buf = [f"{current_clause}. {rest}"] if rest else [f"{current_clause}."]
            continue

        if not buf:
            current_clause = None
        if buf and looks_like_standalone_subheading(line) and _last_line_closes_sentence(buf[-1]):
            flush()
            current_clause = None
        buf.append(line)

    flush()
    return chunks_out

chunks = chunk_by_clauses_stateful(cleaned_text)
print(f"✅ Total clause-based chunks: {len(chunks)}")
if chunks:
    print("First chunk preview:\n", chunks[0]["text"][:500])
    print("\nMetadata:", chunks[0]["metadata"])


✅ Total clause-based chunks: 839
First chunk preview:
 Handbook of Academic Regulations 2025
Owned By: Academic Council
Maintained by: Academic Registrar’s Department
Last updated: April 2025
Approved on: June 2025
Effective from: 1 August 2025
Review date: April 2026

Metadata: {'document': 'Handbook of Academic Regulations 2025–2026', 'part': '', 'section': '', 'clause': '', 'is_continuation': False}


In [9]:


def looks_like_standalone_subheading(line: str) -> bool:
    """Standalone title lines (not Part/Section/clause) — flush chunk so they are not tacked onto the prior clause."""
    s = line.strip()
    if len(s) < 6 or len(s) > 140:
        return False
    if not s[0].isupper():
        return False
    if CLAUSE_LINE.match(s) or PART_LINE.match(s) or SECTION_LINE.match(s):
        return False
    if s.startswith("Note:"):
        return False
    if re.match(r"^Table\s+\d+", s, re.I):
        return False
    if re.match(r"^[a-z]\)\s", s, re.I):
        return False
    if re.match(r"^\d+[-–.)]\s", s):
        return False
    if s.endswith(".") and len(s) > 85:
        return False
    words = s.split()
    if len(words) < 2 or len(words) > 18:
        return False
    if s.count(",") > 2:
        return False
    if len(words) > 8 and re.search(
        r"\b(will|shall|must|should|may|might|can|could|would|is|are|was|were|has|have|had)\b",
        s,
        re.I,
    ):
        return False
    small = {
        "a",
        "an",
        "the",
        "of",
        "in",
        "for",
        "to",
        "and",
        "or",
        "on",
        "at",
        "by",
        "with",
        "via",
        "as",
    }

    def titleish_word(w: str) -> bool:
        w = w.strip("’'\"“”")
        if not w:
            return False
        if w.lower() in small:
            return True
        return w[0].isupper()

    if sum(1 for w in words if titleish_word(w)) / len(words) >= 0.6:
        return True
    if re.match(
        r"^(Claims|Principles|Outcomes|Procedures|Guidance|Fulfilment|Collaborations|"
        r"Assessment|Sitting|Introduction|Definitions|Scope)\b",
        s,
        re.I,
    ):
        return True
    return False


def _last_line_closes_sentence(last: str) -> bool:
    """Avoid splitting mid-clause lists (e.g. document names after a colon) — only flush after a real sentence end."""
    s = last.rstrip()
    if not s:
        return False
    while s and s[-1] in "\"'":
        s = s[:-1].rstrip()
    if not s:
        return False
    if s[-1] in ".!?":
        return True
    if s[-1] == ")" and len(s) >= 2 and s[-2] in ".!?":
        return True
    return False


def _meta_dict(part: str, section: str, clause: str) -> dict:
    """Strings only (Chroma-friendly). Align with Meta_data / ingest: clause + is_continuation."""
    return {
        "document": DOCUMENT_TITLE,
        "part": part or "",
        "section": section or "",
        "clause": clause or "",
        "is_continuation": False,
    }


def chunk_by_clauses_stateful(cleaned: str) -> list:
    current_part = ""
    current_section = ""
    chunks_out = []
    buf: list[str] = []
    current_clause = None  # last clause id for chunk body; None = preamble / non-clause block

    def flush():
        nonlocal buf, current_clause
        if not buf:
            return
        text = "\n".join(buf).strip()
        if not text:
            buf = []
            return
        chunks_out.append(
            {
                "text": text,
                "metadata": _meta_dict(current_part, current_section, current_clause or ""),
            }
        )
        buf = []

    for line in cleaned.splitlines():
        line = line.rstrip()
        if not line:
            continue

        pm = PART_LINE.match(line)
        if pm:
            flush()
            current_part = pm.group(1)
            current_section = ""
            continue

        sm = SECTION_LINE.match(line)
        if sm:
            flush()
            current_section = sm.group(1)
            continue

        cm = CLAUSE_LINE.match(line)
        if cm:
            flush()
            current_clause = cm.group(1)
            rest = cm.group(2).strip()
            buf = [f"{current_clause}. {rest}"] if rest else [f"{current_clause}."]
            continue

        if not buf:
            current_clause = None
        if buf and looks_like_standalone_subheading(line) and _last_line_closes_sentence(buf[-1]):
            flush()
            current_clause = None
        buf.append(line)

    flush()
    return chunks_out


chunks = chunk_by_clauses_stateful(cleaned_text)
print(f"✅ Total clause-based chunks: {len(chunks)}")
if chunks:
    print("First chunk preview:\n", chunks[0]["text"][:500])
    print("\nMetadata:", chunks[0]["metadata"])


✅ Total clause-based chunks: 839
First chunk preview:
 Handbook of Academic Regulations 2025
Owned By: Academic Council
Maintained by: Academic Registrar’s Department
Last updated: April 2025
Approved on: June 2025
Effective from: 1 August 2025
Review date: April 2026

Metadata: {'document': 'Handbook of Academic Regulations 2025–2026', 'part': '', 'section': '', 'clause': '', 'is_continuation': False}


In [10]:
# Step 3 — manual QA: 20 random chunks (seed=42). Check clause isolation, page noise, metadata.
import random

random.seed(42)
k = min(20, len(chunks))
idxs = sorted(random.sample(range(len(chunks)), k)) if chunks else []

for j, i in enumerate(idxs, 1):
    c = chunks[i]
    meta = c["metadata"]
    txt = c["text"]
    has_part_heading = bool(re.search(r"(?:^|\n)Part\s+\d+\s*:", txt))
    has_section_heading = bool(re.search(r"(?:^|\n)Section\s+\d+\s*:", txt))
    lone_page_like = bool(re.search(r"(?:^|\n)\d{1,3}(?:\n|$)", txt))
    print("=" * 72)
    print(f"Sample {j}/{k} | index={i}")
    print(f"  metadata: {meta}")
    print(f"  flags: Part_heading_in_body={has_part_heading} Section_heading_in_body={has_section_heading} lone_digits_line={lone_page_like}")
    print(txt[:700])
    print()



Sample 1/20 | index=25
  metadata: {'document': 'Handbook of Academic Regulations 2025–2026', 'part': '1', 'section': '2', 'clause': '2.11', 'is_continuation': False}
  flags: Part_heading_in_body=False Section_heading_in_body=False lone_digits_line=False
2.11. The University’s regulations and processes for taught courses provide for Dual Awards, or Double Degrees, Joint Awards and Multiple Awards within prevailing legislative and advisory
frameworks of the European Union (EU), UK, and the states of current and potential partner institutions.
Maintenance of academic standards within the University taught courses
G
eneral principles

Sample 2/20 | index=30
  metadata: {'document': 'Handbook of Academic Regulations 2025–2026', 'part': '1', 'section': '2', 'clause': '2.16', 'is_continuation': False}
  flags: Part_heading_in_body=False Section_heading_in_body=False lone_digits_line=False
2.16. In respect of the validation of a course the University will seek to ensure that both the teachin

In [11]:
import json
from pathlib import Path

chunks_path = Path("data/processed") / "academic_handbook_2025_26_chunks.json"
chunks_path.parent.mkdir(parents=True, exist_ok=True)

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("✅ Saved chunks + metadata to:", chunks_path.resolve())

✅ Saved chunks + metadata to: C:\Users\Hasan Jaafar\Documents\finalyearproject\final submission\chatbot\chatbot\data\processed\academic_handbook_2025_26_chunks.json
